# Exploration notebook

Interactive analysis across experiments. Point `STUDY_DIR` at any results folder
(single study or aggregate of symlinked runs).

This notebook is a **template** — edit freely, it is gitignored after the initial commit.
Promote any reusable plot pattern to `analysis/plot.py`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.load import load_results, filter_results, to_dataframe
from analysis.plot import metric_vs_param, convergence_curves, compare_algorithms

STUDY_DIR = '../results/aggregate'   # ← change this
OUTPUT_DIR = '../figs/explore'

PARAMS  = ['algorithm', 'lmbda', 'lmbda_m', 'p', 'q', 'r',
           'scale', 'noise_level', 'sigma_blur', 'max_iter', 'max_iter_cp']
METRICS = ['PSNR_mean', 'SSIM_mean', 'SAM_mean', 'RNMSE_mean', 'CC_mean']

results = load_results(STUDY_DIR)
df = to_dataframe(results)
print(f'{len(df)} experiments loaded')
print(f'Columns: {list(df.columns)}')

## 1. Coverage — what has been run?

Quick table of all experiments with key params and best metric.

In [ ]:
# Full parameter table, sorted by PSNR
cols = [c for c in PARAMS if c in df.columns] + [c for c in METRICS if c in df.columns]
df[cols].sort_values('PSNR_mean', ascending=False)

In [ ]:
# Parameter space coverage: which (lmbda, noise_level) pairs have been explored?
# Adjust x, y, hue, size to the axes you care about.
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='lmbda', y='noise_level',
                hue='algorithm', size='PSNR_mean', sizes=(40, 200),
                ax=ax)
ax.set_xscale('log')
ax.set_title('Explored parameter space')
plt.tight_layout()

## 2. Metrics overview

In [ ]:
# Summary statistics per algorithm
metric_cols = [c for c in METRICS if c in df.columns]
df.groupby('algorithm')[metric_cols].agg(['mean', 'std']).round(3)

In [ ]:
# All metrics at once, one subplot per metric
fig, axes = plt.subplots(1, len(metric_cols), figsize=(4 * len(metric_cols), 4))
for ax, m in zip(axes, metric_cols):
    sns.boxplot(data=df, x='algorithm', y=m, ax=ax)
    ax.set_title(m.replace('_mean', ''))
    ax.set_xlabel('')
plt.suptitle('Metric distributions by algorithm', y=1.02)
plt.tight_layout()

## 3. Parameter impact

Swap `param` and `metric` freely. Use `filter_results` to fix other axes first.

In [ ]:
metric_vs_param(results, param='lmbda',       metric_name='PSNR', output_dir=OUTPUT_DIR)
metric_vs_param(results, param='noise_level', metric_name='PSNR', output_dir=OUTPUT_DIR)

In [ ]:
# Two-parameter interaction: lmbda vs noise_level coloured by PSNR
# (pivot to a heatmap if the grid is regular; scatter otherwise)
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df['lmbda'], df['noise_level'],
                c=df['PSNR_mean'], cmap='viridis', s=80)
plt.colorbar(sc, ax=ax, label='PSNR')
ax.set_xscale('log')
ax.set_xlabel('lmbda'); ax.set_ylabel('noise_level')
ax.set_title('PSNR over (lmbda, noise_level)')
plt.tight_layout()

## 4. Convergence

In [ ]:
# Lines by lmbda, one panel per algorithm
convergence_curves(results, group_by='lmbda', facet_by='algorithm',
                   mode='distance', output_dir=OUTPUT_DIR)

In [ ]:
# Fix one variable, explore another
subset = filter_results(results, noise_level=40)
convergence_curves(subset, group_by='lmbda', facet_by='algorithm',
                   mode='relval', zoom_tail=0.3, output_dir=OUTPUT_DIR)

## 5. Deep dive

Free-form section. Use `filter_results` to isolate a slice, then plot what matters.

In [ ]:
# Example: best run overall
best = df.loc[df['PSNR_mean'].idxmax()]
print(best[[c for c in PARAMS + ['PSNR_mean', 'SAM_mean'] if c in best.index]])

In [ ]:
# Example: GradAlign only, fix norm orders, sweep lmbda
subset = filter_results(results, algorithm='GradAlign', p=2.0, q=2.0, r=1.0)
metric_vs_param(subset, 'lmbda', 'PSNR', OUTPUT_DIR)